# 🌿 Python Backtracking — The Master Guide
### *From Zero to Interview-Ready*

---

> **Mental Model First:**
> Imagine navigating a corn maze. At each fork you pick one direction and walk it to the end. When you hit a dead end, you walk *back* to the last fork and try the next direction. Backtracking works the same way: choose a path, explore it fully, then undo your choice and try the next option. The `pop()` after the recursive call is the walk-back — that one line is the entire secret of the technique.

---

## 📋 Table of Contents

| # | Section |
|---|----------|
| 1 | [What Is Backtracking? The Visual Model](#1) |
| 2 | [Creating / Setup — The Scaffold](#2) |
| 3 | [The Core API — Choose, Explore, Unchoose](#3) |
| 4 | [Decision Map — When To Use What](#4) |
| 5 | [Pattern 1: Permutations (LC 46)](#5) |
| 6 | [Pattern 2: Subsets / Power Set (LC 78)](#6) |
| 7 | [Pattern 3: Combination Sum (LC 39)](#7) |
| 8 | [Pattern 4: N-Queens (LC 51)](#8) |
| 9 | [Pattern 5: Word Search (LC 79)](#9) |
| 10 | [The Backtracking Decision Map](#10) |
| 11 | [Interview Cheat Sheet](#11) |
| 12 | [Summary Map](#12) |

<a id='1'></a>
## 1. 🗺️ What Is Backtracking? The Visual Model

```
DECISION TREE — Permutations of [1, 2, 3]

                              []
                   /           |           \
               [1]            [2]           [3]
              /   \          /   \          /   \
          [1,2] [1,3]    [2,1] [2,3]    [3,1] [3,2]
            |     |        |     |        |     |
         [1,2,3][1,3,2] [2,1,3][2,3,1] [3,1,2][3,2,1]
         ✅SNAP  ✅SNAP   ✅SNAP  ✅SNAP   ✅SNAP  ✅SNAP

PATTERN:  choose one item → recurse deeper → UNDO (pop)
          └── the undo is what "back" means in backtracking

THE UNDO LINE:
  path.append(nums[i])   # CHOOSE
  backtrack(path)         # EXPLORE — goes all the way down the tree
  path.pop()             # UNCHOOSE — walk back to the fork

WITHOUT the pop():
  path grows forever — every branch shares a corrupted state

WITH the pop():
  each branch sees a clean path — isolated from its siblings

TIME COST:   O(n!) — every branch multiplies by remaining choices
SPACE COST:  O(n)  — recursion depth = n, not the full tree width
SNAPSHOT:    result.append(path[:]) — copy before undo corrupts it
```

<a id='2'></a>
## 2. 🔧 Creating / Setup — The Scaffold

In [ ]:
# THE BACKTRACKING SCAFFOLD
# Three shapes: result accumulator, path list, used[] or start index

# Shape 1: used[] boolean array (permutations — order matters)
nums = [1, 2, 3]
used = [False] * len(nums)   # True = this index is already in current path
result = []
path = []

# Shape 2: start index (combinations/subsets — order does NOT matter)
# backtrack(start=0) → only pick indices >= start
# this prevents [2,1] after [1,2] — they are the same combination

# Shape 3: reuse flag (combination sum — can pick same element again)
# recurse with same i  → allow reuse
# recurse with i + 1  → no reuse

# THE UNIVERSAL TEMPLATE:
def backtrack_template(path, start):
    # BASE CASE: path is complete — snapshot it
    if len(path) == len(nums):
        result.append(path[:])      # ALWAYS copy — path is mutable
        return

    for i in range(start, len(nums)):
        if used[i]:
            continue                # already chosen in this path — skip

        # CHOOSE
        path.append(nums[i])
        used[i] = True

        # EXPLORE
        backtrack_template(path, i + 1)   # i+1 for combos, i for reuse allowed

        # UNCHOOSE — restore so next sibling branch starts clean
        path.pop()
        used[i] = False

print("Scaffold shapes:")
print("  used[]     → permutations (order matters, every index each level)")
print("  start idx  → combinations (order doesn't matter, forward only)")
print("  i not i+1  → reuse allowed (combination sum)")
print("Invariant:  CHOOSE → EXPLORE → UNCHOOSE, always in that order")

<a id='3'></a>
## 3. ⚡ The Core API — Choose, Explore, Unchoose

```
OPERATION                      COMPLEXITY   WHAT IT DOES
─────────────────────────────────────────────────────────────────────
path.append(x)                 O(1)         choose — add x to current path
backtrack(...)                 O(varies)    explore — recurse one level deeper
path.pop()                     O(1)         unchoose — erase last choice
result.append(path[:])         O(n)         snapshot — copy before unwinding
used[i] = True / False         O(1)         mark / unmark for permutations
if candidate > remaining: break  O(1)       prune — cut subtree (sorted only)
board[r][c] = '#' / restore    O(1)         in-place mark for grid DFS
─────────────────────────────────────────────────────────────────────

THINGS YOU DO NOT DO:
❌  result.append(path)      path is mutable — all entries will be the same list
✅  result.append(path[:])   always snapshot with a shallow copy
❌  path.pop() before return at base case — pop is always AFTER the recursive call
❌  used[i] check after append — check BEFORE, not inside the recursion
❌  forget to unchoose — the next sibling branch inherits a corrupted path
❌  prune with break on unsorted input — break only works when candidates are sorted
```

In [ ]:
# LIVE DEMO: trace the choose / explore / unchoose cycle on [1, 2]

demo_nums = [1, 2]
demo_result = []
demo_depth = [0]   # track recursion depth for indented output

def backtrack_demo(path, start):
    indent = "  " * demo_depth[0]
    print(f"{indent}ENTER  path={path}  start={start}")

    if len(path) == len(demo_nums):
        snap = path[:]
        demo_result.append(snap)
        print(f"{indent}  BASE CASE → snapshot {snap}")
        return

    for i in range(start, len(demo_nums)):
        # CHOOSE
        path.append(demo_nums[i])
        print(f"{indent}  CHOOSE {demo_nums[i]}  path={path}")
        demo_depth[0] += 1

        # EXPLORE
        backtrack_demo(path, i + 1)

        # UNCHOOSE
        demo_depth[0] -= 1
        removed = path.pop()
        print(f"{indent}  UNCHOOSE {removed}  path={path}")

backtrack_demo([], 0)
print(f"\nAll subsets of [1,2]: {demo_result}")

<a id='4'></a>
## 4. 🗂️ Decision Map — When To Use What

```
SIGNAL IN THE PROBLEM                    SHAPE TO USE
──────────────────────────────────────────────────────────────────
"all permutations" / order matters       used[] boolean, no start index
"all subsets" / power set                start index, advance to i+1
"combinations that sum to target"        start index, allow reuse with i
"place N queens" / constraint grid       row-by-row, conflict sets
"find if word exists in grid"            DFS + in-place mark board[r][c]='#'
"valid parentheses" / balanced strings   open/close counters as state
──────────────────────────────────────────────────────────────────

KEY QUESTION 1: Does order matter?
  YES (permutations) → used[] boolean, iterate ALL indices each level
  NO  (combinations) → start index, iterate from current position forward

KEY QUESTION 2: Can we reuse the same element?
  YES → recurse with same index i (not i+1)     [LC 39 Combination Sum]
  NO  → recurse with i+1                         [LC 40 Combination Sum II]

KEY QUESTION 3: Are there duplicates in input?
  YES → sort first, then: if i > start and nums[i] == nums[i-1]: continue
  NO  → no deduplication needed

PRUNING RULE: only use `break` when candidates are SORTED
  if candidates[i] > remaining: break   # all later candidates also too big
```

<a id='5'></a>
## 5. 🧩 Pattern 1: Permutations — LC 46

---

```
PROBLEM:  Given n distinct integers, return all n! permutations.
APPROACH: At each level, try every index not yet in current path.
          used[] boolean array tracks which indices are live.

SLOW MOTION TRACE on nums = [1, 2]:

  ENTER  path=[]   used=[F,F]
    CHOOSE 1  →  path=[1]  used=[T,F]
      ENTER  path=[1]  used=[T,F]
        CHOOSE 2  →  path=[1,2]  used=[T,T]
          BASE CASE → snapshot [1,2]
        UNCHOOSE 2  →  path=[1]  used=[T,F]
    UNCHOOSE 1  →  path=[]  used=[F,F]
    CHOOSE 2  →  path=[2]  used=[F,T]
      ENTER  path=[2]  used=[F,T]
        CHOOSE 1  →  path=[2,1]  used=[T,T]
          BASE CASE → snapshot [2,1]
        UNCHOOSE 1  →  path=[2]  used=[F,T]
    UNCHOOSE 2  →  path=[]  used=[F,F]
  RESULT: [[1,2], [2,1]]

KEY INSIGHT: used[] is what keeps [1,1] from appearing.
             Without it, each level would pick index 0 again.
TIME:  O(n! * n) — n! permutations, O(n) each to snapshot
SPACE: O(n)      — recursion depth n, used[] size n
```

In [ ]:
def permute(nums):
    """
    LC 46 — Permutations
    Approach: backtracking with used[] to prevent index reuse in current path.
    Args:
        nums (List[int]): n distinct integers.
    Returns:
        List[List[int]]: all n! permutations.
    Time:  O(n! * n) — n! permutations, O(n) to snapshot each
    Space: O(n)      — recursion depth + path + used array
    """
    result = []
    used = [False] * len(nums)

    def backtrack(path):
        # base case: path is a complete permutation
        if len(path) == len(nums):
            result.append(path[:])   # snapshot — path will be mutated on return
            return

        for i in range(len(nums)):
            if used[i]:
                continue             # index already in current path — skip

            # CHOOSE
            used[i] = True
            path.append(nums[i])

            # EXPLORE
            backtrack(path)

            # UNCHOOSE — restore for next sibling
            path.pop()
            used[i] = False

    backtrack([])
    return result

# Slow motion on nums = [1, 2, 3]:
# depth 0: try i=0(1) → used=[T,F,F] path=[1]
#   depth 1: try i=1(2) → used=[T,T,F] path=[1,2]
#     depth 2: try i=2(3) → used=[T,T,T] path=[1,2,3] → BASE CASE
#   depth 1: try i=2(3) → used=[T,F,T] path=[1,3]
#     depth 2: try i=1(2) → used=[T,T,T] path=[1,3,2] → BASE CASE
# depth 0: try i=1(2) → starts [2,...] subtree
# depth 0: try i=2(3) → starts [3,...] subtree

def test_harness(fn):
    tests = [
        ([1, 2], sorted([[1, 2], [2, 1]])),
        ([1, 2, 3], sorted([[1,2,3],[1,3,2],[2,1,3],[2,3,1],[3,1,2],[3,2,1]])),
        ([1], [[1]]),
        ([0, 1], sorted([[0, 1], [1, 0]])),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = sorted(fn(*inputs))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(permute)
print("permute defined.")

<a id='6'></a>
## 6. 🧩 Pattern 2: Subsets / Power Set — LC 78

---

```
PROBLEM:  Given n distinct integers, return all 2^n subsets (power set).
APPROACH: At each index, decide: include it or skip it.
          start index advances to i+1, so we never pick earlier elements again.
          Snapshot at EVERY level, not just when path is full.

SLOW MOTION TRACE on nums = [1, 2]:

  ENTER start=0  path=[] → snapshot []
    CHOOSE 1  path=[1]
      ENTER start=1  path=[1] → snapshot [1]
        CHOOSE 2  path=[1,2]
          ENTER start=2  path=[1,2] → snapshot [1,2]  (loop empty)
        UNCHOOSE 2  path=[1]
    UNCHOOSE 1  path=[]
    CHOOSE 2  path=[2]
      ENTER start=2  path=[2] → snapshot [2]  (loop empty)
    UNCHOOSE 2  path=[]
  RESULT: [[], [1], [1,2], [2]]

KEY INSIGHT: Snapshotting at entry (before the loop) captures every subset.
             start index prevents [2,1] after [1,2] — same subset, different order.
TIME:  O(2^n * n) — 2^n subsets, O(n) each to snapshot
SPACE: O(n)       — max recursion depth = n
```

In [ ]:
def subsets(nums):
    """
    LC 78 — Subsets
    Approach: backtracking with start index; snapshot at entry of every call.
    Args:
        nums (List[int]): n distinct integers.
    Returns:
        List[List[int]]: all 2^n subsets.
    Time:  O(2^n * n) — 2^n subsets, O(n) each to snapshot
    Space: O(n)       — recursion depth, path size
    """
    result = []

    def backtrack(start, path):
        result.append(path[:])   # snapshot at entry — this path is a valid subset

        for i in range(start, len(nums)):
            # CHOOSE: include nums[i]
            path.append(nums[i])

            # EXPLORE: only consider elements after i — no going backwards
            backtrack(i + 1, path)

            # UNCHOOSE: exclude nums[i] for next sibling
            path.pop()

    backtrack(0, [])
    return result

# Slow motion on nums = [1, 2, 3]:
# call(0,[])  → snap []  → try 1
#   call(1,[1]) → snap [1] → try 2
#     call(2,[1,2]) → snap [1,2] → try 3
#       call(3,[1,2,3]) → snap [1,2,3] → loop empty → return
#     pop 3 → [1,2], loop ends
#   pop 2 → [1], try 3
#     call(3,[1,3]) → snap [1,3] → loop empty → return
#   pop 3 → [1], loop ends
# pop 1 → [], try 2 ...
# total 8 subsets = 2^3

def test_harness(fn):
    tests = [
        ([1, 2, 3], sorted([[], [1], [2], [3], [1,2], [1,3], [2,3], [1,2,3]])),
        ([1, 2], sorted([[], [1], [2], [1,2]])),
        ([0], sorted([[], [0]])),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = sorted(fn(*inputs))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(subsets)
print("subsets defined.")

<a id='7'></a>
## 7. 🧩 Pattern 3: Combination Sum — LC 39

---

```
PROBLEM:  Given candidates (distinct positive ints) and a target, return all
          unique combinations where the chosen numbers sum to target.
          Each number may be used UNLIMITED times.
APPROACH: Sort candidates first. Use start index. Recurse with SAME i (not i+1)
          to allow reuse. Prune with `break` when candidate > remaining.

SLOW MOTION TRACE on candidates=[2,3,6,7], target=7:
  (candidates already sorted)

  call(start=0, rem=7, path=[])
    try 2: call(start=0, rem=5, path=[2])
      try 2: call(start=0, rem=3, path=[2,2])
        try 2: call(start=0, rem=1, path=[2,2,2])
          try 2: rem=1-2=-1 < 0 → break  (sorted, so rest also too big)
        try 3: call(start=1, rem=0, path=[2,2,3]) → BASE CASE → snap [2,2,3]
        try 6: 3<6 → break
      try 3: call(start=1, rem=2, path=[2,3])
        try 3: rem=2-3=-1 < 0 → break
      try 6: 5<6 → break
    try 3: call(start=1, rem=4, path=[3])
      try 3: call(start=1, rem=1, path=[3,3])
        try 3: -2 < 0 → break
      try 6: 4<6 → break
    try 6: call(start=2, rem=1, path=[6])
      try 6: -5 < 0 → break
    try 7: call(start=3, rem=0, path=[7]) → BASE CASE → snap [7]
  RESULT: [[2,2,3], [7]]

KEY INSIGHT: Recurse with same i (not i+1) = allow reuse.
             Sort + break = prune entire subtree when candidate exceeds remaining.
TIME:  O(N^(T/M)) — N candidates, T target, M smallest candidate
SPACE: O(T/M)     — max recursion depth
```

In [ ]:
def combination_sum(candidates, target):
    """
    LC 39 — Combination Sum
    Approach: sort, backtrack with same index i for reuse, break when over target.
    Args:
        candidates (List[int]): distinct positive integers.
        target (int): sum to reach.
    Returns:
        List[List[int]]: all unique combinations summing to target.
    Time:  O(N^(T/M)) — N candidates, T target, M smallest candidate
    Space: O(T/M)     — max recursion depth = target / min_candidate
    """
    candidates.sort()   # sort enables pruning with break
    result = []

    def backtrack(start, remaining, path):
        if remaining == 0:
            result.append(path[:])   # exact hit — snapshot
            return

        for i in range(start, len(candidates)):
            c = candidates[i]
            if c > remaining:
                break               # sorted — all remaining candidates also too big

            # CHOOSE
            path.append(c)

            # EXPLORE — recurse with i (same index) to allow reuse of c
            backtrack(i, remaining - c, path)

            # UNCHOOSE
            path.pop()

    backtrack(0, target, [])
    return result

# Slow motion on candidates=[2,3,6,7], target=7:
# path=[2,2,3] found: 2+2+3=7 ✓
# path=[7] found: 7=7 ✓
# path=[2,2,2,?] pruned: 2+2+2=6, next candidate 2 → rem=1, 2>1 → break
# path=[3,3,?] pruned: 3+3=6, next 3 → rem=1, 3>1 → break

def test_harness(fn):
    tests = [
        ([2, 3, 6, 7], 7, sorted([[2,2,3],[7]])),
        ([2, 3, 5], 8, sorted([[2,2,2,2],[2,3,3],[3,5]])),
        ([2], 1, []),                                          # no solution
        ([1], 2, [[1, 1]]),
    ]
    passed = 0
    for *inputs, expected in tests:
        got = sorted(fn(*inputs))
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | input={inputs} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(combination_sum)
print("combination_sum defined.")

<a id='8'></a>
## 8. 🧩 Pattern 4: N-Queens — LC 51

---

```
PROBLEM:  Place n queens on an n×n chessboard so no two attack each other.
          Return all distinct solutions as board configurations.
APPROACH: Place one queen per row. Track forbidden columns and diagonals with sets.
          For a queen at (row, col):
            - Diagonal    (top-left → bottom-right): row - col is constant
            - Anti-diagonal (top-right → bottom-left): row + col is constant

SLOW MOTION TRACE on n=4, row 0:

  row=0: try col=0 → cols={0} diag={0} anti={0}
    row=1: col=0 in cols → skip
    row=1: col=1 diag=0 in diag → skip
    row=1: col=2 → safe → cols={0,2} diag={0,-1} anti={0,3}
      row=2: col=0 in cols → skip
      row=2: col=1 → diag=2-1=1, anti=2+1=3 in anti → skip
      row=2: col=2 in cols → skip
      row=2: col=3 → diag=2-3=-1 in diag → skip
    row=1: col=3 → safe → ...
      row=2: col=1 → safe ... → row=3: eventually finds solution

  Two solutions for n=4:
    [".Q..","...Q","Q...","..Q."]  and  ["..Q.","Q...","...Q",".Q.."]

KEY INSIGHT: Three sets (cols, diag, anti_diag) replace the O(n²) conflict check.
             Placing one queen per row guarantees no row conflicts automatically.
TIME:  O(n!) — worst case tries n! placements
SPACE: O(n)  — 3 sets + board build O(n²) at snapshot time
```

In [ ]:
def solve_n_queens(n):
    """
    LC 51 — N-Queens
    Approach: one queen per row, conflict sets for O(1) validity check.
    Args:
        n (int): board dimension and queen count.
    Returns:
        List[List[str]]: all valid board configurations.
    Time:  O(n!) — number of valid placements explored
    Space: O(n)  — three sets + recursion depth; board built only at base case
    """
    result = []
    cols = set()       # columns with a queen
    diag = set()       # (row - col) constant along top-left→bottom-right diagonal
    anti_diag = set()  # (row + col) constant along top-right→bottom-left diagonal
    board = ["." * n for _ in range(n)]   # mutable list for current placement
    board = [list("." * n) for _ in range(n)]   # list of lists for mutability

    def backtrack(row):
        if row == n:
            # build string snapshot of current board
            result.append(["".join(r) for r in board])
            return

        for col in range(n):
            d = row - col     # diagonal key
            a = row + col     # anti-diagonal key

            if col in cols or d in diag or a in anti_diag:
                continue      # queen would attack another — skip

            # CHOOSE: place queen at (row, col)
            cols.add(col)
            diag.add(d)
            anti_diag.add(a)
            board[row][col] = "Q"

            # EXPLORE: place queen in next row
            backtrack(row + 1)

            # UNCHOOSE: remove queen, free the constraints
            cols.remove(col)
            diag.remove(d)
            anti_diag.remove(a)
            board[row][col] = "."

    backtrack(0)
    return result

# Slow motion on n=4, first solution found:
# row=0 col=1: place Q → board[0]=".Q.."
# row=1 col=3: place Q → board[1]="...Q"
# row=2 col=0: place Q → board[2]="Q..."
# row=3 col=2: place Q → board[3]="..Q." → BASE CASE → snap

def test_harness(fn):
    tests = [
        (1, [["Q"]]),
        (2, []),                         # no solution for 2×2
        (3, []),                         # no solution for 3×3
        (4, 2),                          # 2 solutions for 4×4 (check count)
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        if isinstance(expected, int):
            ok = len(got) == expected
            status = "PASSED" if ok else "FAILED"
            if not ok:
                print(f"FAILED | n={inputs} | expected {expected} solutions | got {len(got)}")
        else:
            ok = sorted(got) == sorted(expected)
            status = "PASSED" if ok else "FAILED"
            if not ok:
                print(f"FAILED | n={inputs} | expected={expected} | got={got}")
        passed += ok
    print(f"{passed}/{len(tests)} tests passed")

test_harness(solve_n_queens)
print(solve_n_queens(4))
print("solve_n_queens defined.")

<a id='9'></a>
## 9. 🧩 Pattern 5: Word Search — LC 79

---

```
PROBLEM:  Given a 2D character grid and a word, return True if the word
          can be traced by adjacent (4-directional) cells without reuse.
APPROACH: Try DFS from every cell matching word[0]. Mark visited cells
          in-place with '#', restore after backtracking.

SLOW MOTION TRACE on board:
  A B C E
  S F C S
  A D E E
  word = "ABCCED"

  Start at (0,0)='A', word[0]='A' ✓
    mark (0,0)='#',  try neighbors for 'B'
    → (0,1)='B' ✓, mark (0,1)='#', try neighbors for 'C'
      → (0,2)='C' ✓, mark (0,2)='#', try neighbors for 'C'
        → (1,2)='C' ✓, mark (1,2)='#', try neighbors for 'E'
          → (2,2)='E' ✓, mark (2,2)='#', try neighbors for 'D'
            → (2,1)='D' ✓, mark (2,1)='#' → word[6]=end → return True
  Path: (0,0)→(0,1)→(0,2)→(1,2)→(2,2)→(2,1) = "ABCCED" ✓

KEY INSIGHT: board[r][c]='#' serves as both the mark AND the restore hook.
             The original char is saved before marking, then restored after.
             Early return True propagates up without more exploration.
TIME:  O(R*C * 4^L) — start from every cell, DFS depth = word length L
SPACE: O(L)         — recursion depth = word length
```

In [ ]:
def exist(board, word):
    """
    LC 79 — Word Search
    Approach: DFS from each matching start cell; mark visited in-place with '#'.
    Args:
        board (List[List[str]]): 2D grid of characters.
        word (str): target word.
    Returns:
        bool: True if word can be traced on the board.
    Time:  O(R*C * 4^L) — R*C start points, 4-directional DFS to depth L
    Space: O(L)         — recursion stack depth equals word length
    """
    ROWS, COLS = len(board), len(board[0])
    DIRS = [(0, 1), (0, -1), (1, 0), (-1, 0)]   # right, left, down, up

    def dfs(r, c, idx):
        if idx == len(word):
            return True       # matched all characters — found the word
        if r < 0 or r >= ROWS or c < 0 or c >= COLS:
            return False      # out of bounds
        if board[r][c] != word[idx]:
            return False      # character mismatch — wrong branch
        if board[r][c] == "#":
            return False      # already used in this path — no revisit

        # CHOOSE: mark cell as visited
        saved = board[r][c]
        board[r][c] = "#"

        # EXPLORE: try all 4 neighbors for next character
        found = any(dfs(r + dr, c + dc, idx + 1) for dr, dc in DIRS)

        # UNCHOOSE: restore cell for other search paths
        board[r][c] = saved

        return found

    # try every cell as a starting point
    for r in range(ROWS):
        for c in range(COLS):
            if dfs(r, c, 0):
                return True   # found from this starting cell

    return False

# Slow motion on board above, word="ABCCED":
# dfs(0,0,0): board[0][0]='A'==word[0] → mark '#', try neighbors
#   dfs(0,1,1): board[0][1]='B'==word[1] → mark '#', try neighbors
#     dfs(0,2,2): board[0][2]='C'==word[2] → mark '#', try neighbors
#       dfs(1,2,3): board[1][2]='C'==word[3] → mark '#', try neighbors
#         dfs(2,2,4): board[2][2]='E'==word[4] → mark '#', try neighbors
#           dfs(2,1,5): board[2][1]='D'==word[5] → mark '#'
#             dfs(?,?,6): idx==len(word) → return True ✓

def test_harness(fn):
    board1 = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
    board2 = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
    board3 = [["A","B","C","E"],["S","F","C","S"],["A","D","E","E"]]
    board4 = [["a","a"]]
    tests = [
        (board1, "ABCCED", True),
        (board2, "SEE", True),
        (board3, "ABCB", False),    # can't reuse B at (0,1)
        (board4, "aaa", False),     # only 2 cells, need 3
    ]
    passed = 0
    for *inputs, expected in tests:
        got = fn(*inputs)
        status = "PASSED" if got == expected else "FAILED"
        if status == "FAILED":
            print(f"FAILED | word={inputs[1]} | expected={expected} | got={got}")
        passed += (got == expected)
    print(f"{passed}/{len(tests)} tests passed")

test_harness(exist)
print("exist defined.")

<a id='10'></a>
## 10. 🗺️ The Backtracking Decision Map

```
QUESTION TYPE                    KEY TECHNIQUE              LC PROBLEMS
────────────────────────────────────────────────────────────────────────────
All orderings (order matters)    used[] boolean             46 Permutations
All subsets / power set          start index, snap always   78 Subsets
Combinations summing to target   start index, allow reuse   39 Combination Sum
Constraint satisfaction grid     sets for conflicts          51 N-Queens
Path exists in 2D grid           DFS + in-place mark        79 Word Search
────────────────────────────────────────────────────────────────────────────

SHAPE SELECTOR:

  Order matters?  ──YES──► used[] bool, iterate ALL indices each level
       │
      NO
       │
  Can reuse?  ────YES──► start index, recurse with i (same index)
       │
      NO
       │
  Duplicates in input?  ──YES──► sort + if nums[i]==nums[i-1] and i>start: skip
       │
      NO
       │
  Standard combination: start index, recurse with i+1

PRUNING OPPORTUNITIES:
  Sort + break         → cut subtree when candidate exceeds remaining (LC 39)
  Conflict sets        → O(1) validity check vs O(n) linear scan (LC 51)
  in-place mark + idx  → no visited set needed for grid DFS (LC 79)
  len check early      → reject paths longer than allowed before going deeper
```

<a id='11'></a>
## 11. 📋 Interview Cheat Sheet

### When to reach for Backtracking:

| Signal | What to Do |
|--------|------------|
| "all permutations" | used[] boolean, iterate all indices |
| "all subsets" / "power set" | start index, snapshot at entry |
| "combinations summing to X" | start index, reuse allowed or not |
| "place N pieces" / constraint board | row-by-row + conflict sets |
| "find path in grid" | DFS + in-place mark, restore after |
| "generate all valid strings" | open/close counters as state |

### The O(1) operations — memorize these:

```python
path.append(x)          # choose
backtrack(...)          # explore
path.pop()              # unchoose  ← this IS backtracking
result.append(path[:])  # snapshot  ← ALWAYS copy, never reference
board[r][c] = '#'       # grid mark
board[r][c] = saved     # grid restore
```

### Common templates:

```python
# TEMPLATE 1: PERMUTATIONS (order matters, no reuse)
used = [False] * n
def bt(path):
    if len(path) == n: result.append(path[:]); return
    for i in range(n):
        if used[i]: continue
        used[i] = True; path.append(nums[i])
        bt(path)
        path.pop(); used[i] = False

# TEMPLATE 2: SUBSETS (no order, no reuse)
def bt(start, path):
    result.append(path[:])
    for i in range(start, n):
        path.append(nums[i]); bt(i+1, path); path.pop()

# TEMPLATE 3: COMBINATION SUM (no order, reuse allowed)
candidates.sort()
def bt(start, rem, path):
    if rem == 0: result.append(path[:]); return
    for i in range(start, len(candidates)):
        if candidates[i] > rem: break
        path.append(candidates[i]); bt(i, rem-candidates[i], path); path.pop()

# TEMPLATE 4: GRID WORD SEARCH
def dfs(r, c, idx):
    if idx == len(word): return True
    if out_of_bounds or board[r][c] != word[idx]: return False
    saved = board[r][c]; board[r][c] = '#'
    found = any(dfs(r+dr, c+dc, idx+1) for dr,dc in DIRS)
    board[r][c] = saved
    return found
```

### Gotchas to not forget:

```
❌  result.append(path)     — all results will be the same empty list after unwinding
✅  result.append(path[:])  — shallow copy captures the current state
❌  forget path.pop()       — path grows without bound, results are wrong
❌  use break on unsorted input — break prunes based on order, only valid if sorted
✅  sort candidates before pruning with break
❌  mark visited with a set in grid DFS — slower; in-place mark is O(1)
✅  save and restore board[r][c] — cleaner than a separate visited set
❌  snapshot inside the loop body for subsets — miss the empty subset
✅  snapshot at function entry for subsets — captures every valid partial path
```

<a id='12'></a>
## 12. 🗺️ Summary Map

```
                         BACKTRACKING
                              │
                 ┌────────────┼────────────┐
                 │            │            │
           ORDER MATTERS   NO ORDER     GRID
           (Permutations)  (Subsets /   (Word
                │          Combos)      Search)
           used[] bool      │              │
           iterate ALL    start idx     DFS + '#'
           indices        advance       mark/restore
           LC 46          i+1           LC 79
                              │
                   ┌──────────┴──────────┐
                REUSE OK            NO REUSE
                (Combo Sum)         (Subsets)
                recurse i           recurse i+1
                sort + break        LC 78
                LC 39

          CONSTRAINT SATISFACTION
          (N-Queens, Sudoku)
          ─────────────────────
          One queen per row
          cols / diag / anti_diag sets
          O(1) conflict check
          LC 51

THE THREE LINES THAT ARE BACKTRACKING:
  path.append(x)    # CHOOSE
  backtrack(...)    # EXPLORE
  path.pop()        # UNCHOOSE  ← this line IS the backtrack
```

---
*End of Backtracking Master Guide — Sean Edition*